In [62]:
import json
import pandas as pd
import re

def load_eval_jsonl(path):
    """Load HuggingFace eval_predictions.jsonl into a Pandas dataframe."""
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)

    df["correct"] = df["label"] == df["predicted_label"]
    df["incorrect"] = ~df["correct"]

    return df

def count_keyword_errors(df, keyword, field="hypothesis", print_results=True):
    """
    Count how many incorrect examples contain `keyword` 
    in the specified field.
    """
    correct = df[df["correct"] == True]
    correct_matches = correct[correct[field].str.contains(keyword, case=False)]
    incorrect = df[df["incorrect"] == True]
    incorrect_matches = incorrect[incorrect[field].str.contains(keyword, case=False)]
    results = {
        "keyword": keyword,
        "total": len(df),
        "total_incorrect": len(incorrect),
        "correct_with_keyword": len(correct_matches),
        "incorrect_with_keyword": len(incorrect_matches),
        "fraction_correct": len(correct_matches) / max(len(correct), 1),
        "fraction_incorrect": len(incorrect_matches) / max(len(incorrect), 1),
        "fraction_incorrect_correct": len(incorrect_matches) / len(correct_matches)
    }
    if print_results:
        print(results)
    return results


def error_pattern(df, pattern, field="hypothesis", print_results=True):
    """Return all examples matching a regex pattern in a given field."""
    regex = re.compile(pattern, re.IGNORECASE)
    correct = df[df["correct"] == True]
    correct_matches = correct[correct[field].str.contains(regex)]
    incorrect = df[df["incorrect"] == True]
    incorrect_matches = incorrect[incorrect[field].str.contains(regex)]
    results = {
        "pattern": regex,
        "total": len(df),
        "total_incorrect": len(incorrect),
        "correct_with_keyword": len(correct_matches),
        "incorrect_with_keyword": len(incorrect_matches),
        "fraction_correct": len(correct_matches) / max(len(correct), 1),
        "fraction_incorrect": len(incorrect_matches) / max(len(incorrect), 1),
        "fraction_incorrect_correct": len(incorrect_matches) / len(correct_matches)
    }
    if print_results:
        print(results)
    return results

def summary_for_keywords(df, keywords, field="hypothesis", print_results=True):
    results = []
    for kw in keywords:
        stats = count_keyword_errors(df, kw, field, print_results=False)
        results.append(stats)
    df = pd.DataFrame(results).sort_values("fraction_incorrect_correct", ascending=False)
    if print_results:
        print(df)
    return df

def confusion(df, print_results=True):
    """
    Return a confusion matrix dataframe:
    rows = true labels
    columns = predicted labels
    """
    conf = pd.crosstab(df["label"], df["predicted_label"], rownames=["true"], colnames=["pred"])
    if print_results:
        print(conf)
    return conf

In [68]:
baseline = load_eval_jsonl("20251121_055142-anli_test_baseline-1/eval_predictions.jsonl") # misunderstood-cloud-76
weighted_v1 = load_eval_jsonl("20251121_055405-anli_test_weighted-1/eval_predictions.jsonl") # vocal-dragon-77
weighted_v2 = load_eval_jsonl("20251130_112907-anli_test_weighted-1/eval_predictions.jsonl") #silvery-sound-89
weighted_v3 = load_eval_jsonl("20251123_044443-anli_test_weighted-1/eval_predictions.jsonl") # dashing-spaceship-88

# count_keyword_errors(baseline, "not", field="hypothesis")

# error_pattern(baseline, r"\bnot\b")
# error_pattern(baseline, r"\b\d+\b")     # numbers
# error_pattern(baseline, r"\b(?:always|never)\b")
# error_pattern(baseline, r"\b[A-Z][a-z]+ (in|from)\b")  # location phrases

keywords = ["not", "no", "never", "without", "more", "less", "because", "if", "all", "some"]
summary_for_keywords(baseline, keywords, print_results=False)

,keyword,total,total_incorrect,correct_with_keyword,incorrect_with_keyword,fraction_correct,fraction_incorrect,fraction_incorrect_correct
5,less,16946,5848,95,113,0.008560,0.019323,1.189474
0,not,16946,5848,488,359,0.043972,0.061389,0.735656
1,no,16946,5848,1676,1005,0.151018,0.171854,0.599642
8,all,16946,5848,1036,587,0.093350,0.100376,0.566602
4,more,16946,5848,383,198,0.034511,0.033858,0.516971
7,if,16946,5848,517,254,0.046585,0.043434,0.491296
2,never,16946,5848,167,77,0.015048,0.013167,0.461078
9,some,16946,5848,101,39,0.009101,0.006669,0.386139
3,without,16946,5848,11,3,0.000991,0.000513,0.272727
6,because,16946,5848,86,18,0.007749,0.003078,0.209302


In [69]:
summary_for_keywords(weighted_v1, keywords, print_results=False)

,keyword,total,total_incorrect,correct_with_keyword,incorrect_with_keyword,fraction_correct,fraction_incorrect,fraction_incorrect_correct
5,less,16946,6035,105,103,0.009623,0.017067,0.980952
0,not,16946,6035,478,369,0.043809,0.061143,0.771967
1,no,16946,6035,1672,1009,0.153240,0.167191,0.603469
8,all,16946,6035,1028,595,0.094217,0.098592,0.578794
7,if,16946,6035,496,275,0.045459,0.045568,0.554435
4,more,16946,6035,376,205,0.034461,0.033969,0.545213
2,never,16946,6035,167,77,0.015306,0.012759,0.461078
3,without,16946,6035,10,4,0.000917,0.000663,0.400000
9,some,16946,6035,103,37,0.009440,0.006131,0.359223
6,because,16946,6035,88,16,0.008065,0.002651,0.181818


In [70]:
summary_for_keywords(weighted_v2, keywords, print_results=False)

,keyword,total,total_incorrect,correct_with_keyword,incorrect_with_keyword,fraction_correct,fraction_incorrect,fraction_incorrect_correct
5,less,16946,6092,103,105,0.009490,0.017236,1.019417
0,not,16946,6092,479,368,0.044131,0.060407,0.768267
1,no,16946,6092,1673,1008,0.154137,0.165463,0.602510
8,all,16946,6092,1017,606,0.093698,0.099475,0.595870
4,more,16946,6092,366,215,0.033720,0.035292,0.587432
7,if,16946,6092,497,274,0.045790,0.044977,0.551308
2,never,16946,6092,167,77,0.015386,0.012640,0.461078
9,some,16946,6092,102,38,0.009397,0.006238,0.372549
3,without,16946,6092,11,3,0.001013,0.000492,0.272727
6,because,16946,6092,85,19,0.007831,0.003119,0.223529


In [71]:
summary_for_keywords(weighted_v3, keywords, print_results=False)

,keyword,total,total_incorrect,correct_with_keyword,incorrect_with_keyword,fraction_correct,fraction_incorrect,fraction_incorrect_correct
5,less,16946,6084,101,107,0.009298,0.017587,1.059406
0,not,16946,6084,489,358,0.045019,0.058843,0.732106
8,all,16946,6084,1003,620,0.092340,0.101907,0.618146
1,no,16946,6084,1659,1022,0.152734,0.167982,0.616034
7,if,16946,6084,493,278,0.045388,0.045694,0.563895
4,more,16946,6084,381,200,0.035076,0.032873,0.524934
9,some,16946,6084,98,42,0.009022,0.006903,0.428571
2,never,16946,6084,173,71,0.015927,0.011670,0.410405
3,without,16946,6084,11,3,0.001013,0.000493,0.272727
6,because,16946,6084,84,20,0.007733,0.003287,0.238095


In [ ]:
print("baseline accuracy", baseline['correct'].mean() * 100)
print("weighted_v1 accuracy", weighted_v1['correct'].mean() * 100)
print("weighted_v2 accuracy", weighted_v2['correct'].mean() * 100)
print("weighted_v3 accuracy", weighted_v3['correct'].mean() * 100)

baseline accuracy 65.49038121090523
weighted_v1 accuracy 64.38687595892836
weighted_v2 accuracy 64.05051339549156
weighted_v3 accuracy 64.0977221763248
